<a href="https://colab.research.google.com/github/anushia3/road-segmentation-project/blob/main/road_segmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!unzip "/content/drive/MyDrive/road_segmentation/dataset.zip" -d /content/

In [ ]:
import os

print("Train Images:", len(os.listdir("/content/dataset/train/images")))
print("Train Masks:", len(os.listdir("/content/dataset/train/masks")))
print("Val Images:", len(os.listdir("/content/dataset/val/images")))
print("Val Masks:", len(os.listdir("/content/dataset/val/masks")))

In [ ]:
!pip install -q segmentation-models-pytorch
!pip install -q albumentations

In [ ]:
import torch

print("PyTorch Version:", torch.__version__)
print("GPU Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import os
import cv2
import torch
import numpy as np

from torch.utils.data import Dataset

class RoadDataset(Dataset):

    def __init__(self, image_dir, mask_dir):

        self.image_dir = image_dir
        self.mask_dir = mask_dir

        self.images = sorted(os.listdir(image_dir))

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):

        image_name = self.images[idx]

        image_path = os.path.join(
            self.image_dir,
            image_name
        )

        mask_path = os.path.join(
            self.mask_dir,
            image_name
        )

        image = cv2.imread(image_path)
        image = cv2.cvtColor(
            image,
            cv2.COLOR_BGR2RGB
        )

        mask = cv2.imread(
            mask_path,
            cv2.IMREAD_GRAYSCALE
        )

        image = image.astype(np.float32) / 255.0

        mask = mask.astype(np.float32) / 255.0

        image = torch.tensor(
            image
        ).permute(2,0,1)

        mask = torch.tensor(
            mask
        ).unsqueeze(0)

        return image, mask

In [ ]:
train_dataset = RoadDataset(
    "/content/dataset/train/images",
    "/content/dataset/train/masks"
)

val_dataset = RoadDataset(
    "/content/dataset/val/images",
    "/content/dataset/val/masks"
)

print("Train samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))

In [ ]:
!ls /content

In [ ]:
!find /content -type d | head -50

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!find "/content/drive/MyDrive" -name "*.zip"

In [ ]:
!find /content -type d | grep dataset

In [ ]:
train_dataset = RoadDataset(
    "/content/dataset/train/images",
    "/content/dataset/train/masks"
)

val_dataset = RoadDataset(
    "/content/dataset/val/images",
    "/content/dataset/val/masks"
)

print("Train samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))

In [ ]:
image, mask = train_dataset[0]

print("Image shape:", image.shape)
print("Mask shape:", mask.shape)

print("Image min:", image.min())
print("Image max:", image.max())

print("Mask min:", mask.min())
print("Mask max:", mask.max())

In [ ]:
import matplotlib.pyplot as plt

image, mask = train_dataset[0]

image = image.permute(1, 2, 0).numpy()
mask = mask.squeeze().numpy()

plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.imshow(image)
plt.title("Image")

plt.subplot(1,2,2)
plt.imshow(mask, cmap="gray")
plt.title("Mask")

plt.show()

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=2
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=2
)

In [ ]:
images, masks = next(iter(train_loader))

print("Images:", images.shape)
print("Masks:", masks.shape)

In [ ]:
import segmentation_models_pytorch as smp

model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1
)

model = model.cuda()

print("Model ready")

In [ ]:
!pip install -q segmentation-models-pytorch

In [ ]:
import segmentation_models_pytorch as smp

print(smp.__version__)
model = smp.Unet(
    encoder_name="resnet50",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1
)

In [ ]:
import segmentation_models_pytorch as smp
import torch

model = smp.Unet(
    encoder_name="resnet50",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1
)

device = torch.device("cuda")

model = model.to(device)

print("Model ready")

In [ ]:
images, masks = next(iter(train_loader))

images = images.to(device)

with torch.no_grad():
    preds = model(images)

print("Input shape :", images.shape)
print("Output shape:", preds.shape)

In [ ]:
import torch.nn as nn

dice_loss = smp.losses.DiceLoss(
    mode="binary"
)

bce_loss = nn.BCEWithLogitsLoss()

def loss_fn(preds, masks):
    return (
        dice_loss(preds, masks)
        + bce_loss(preds, masks)
    )

In [ ]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=3e-4
)

In [ ]:
images, masks = next(iter(train_loader))

images = images.to(device)
masks = masks.to(device)

optimizer.zero_grad()

preds = model(images)

loss = loss_fn(preds, masks)

loss.backward()

optimizer.step()

print("Loss:", loss.item())

In [ ]:
from tqdm import tqdm

EPOCHS = 20

for epoch in range(EPOCHS):

    model.train()

    running_loss = 0

    loop = tqdm(train_loader)

    for images, masks in loop:

        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()

        preds = model(images)

        loss = loss_fn(preds, masks)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        loop.set_description(
            f"Epoch {epoch+1}/{EPOCHS}"
        )

        loop.set_postfix(
            loss=loss.item()
        )

    avg_loss = running_loss / len(train_loader)

    print(
        f"\nEpoch {epoch+1} Average Loss: {avg_loss:.4f}"
    )

In [ ]:
model.eval()

val_loss = 0

with torch.no_grad():

    for images, masks in val_loader:

        images = images.to(device)
        masks = masks.to(device)

        preds = model(images)

        loss = loss_fn(preds, masks)

        val_loss += loss.item()

val_loss /= len(val_loader)

print("Validation Loss:", val_loss)

In [ ]:
import matplotlib.pyplot as plt
import torch
import random

model.eval()

# Select a random index from the validation dataset
random_index = random.randint(0, len(val_dataset) - 1)
image, mask = val_dataset[random_index]

with torch.no_grad():

    pred = model(
        image.unsqueeze(0).to(device)
    )

pred = torch.sigmoid(pred)

pred = pred.squeeze().cpu().numpy()

image = image.permute(1,2,0).numpy()
mask = mask.squeeze().numpy()

pred_binary = (pred > 0.5).astype(float)

plt.figure(figsize=(15,5))

plt.subplot(1,3,1)
plt.imshow(image)
plt.title("Satellite Image")

plt.subplot(1,3,2)
plt.imshow(mask, cmap="gray")
plt.title("Ground Truth")

plt.subplot(1,3,3)
plt.imshow(pred_binary, cmap="gray")
plt.title("Prediction")

plt.show()

In [ ]:
torch.save(
    model.state_dict(),
    "/content/drive/MyDrive/road_segmentation/unet_roads.pth"
)

print("Model saved")

In [ ]:
import numpy as np

model.eval()

ious = []

with torch.no_grad():

    for images, masks in val_loader:

        images = images.to(device)
        masks = masks.to(device)

        preds = model(images)

        preds = torch.sigmoid(preds)

        preds = (preds > 0.5).float()

        intersection = (
            preds * masks
        ).sum((1,2,3))

        union = (
            preds + masks
        ).clamp(0,1).sum((1,2,3))

        iou = (
            intersection + 1e-6
        ) / (
            union + 1e-6
        )

        ious.extend(
            iou.cpu().numpy()
        )

print(
    "Mean IoU:",
    np.mean(ious)
)

In [ ]:
import numpy as np

model.eval()

dice_scores = []

with torch.no_grad():

    for images, masks in val_loader:

        images = images.to(device)
        masks = masks.to(device)

        preds = model(images)

        preds = torch.sigmoid(preds)
        preds = (preds > 0.5).float()

        intersection = (
            preds * masks
        ).sum((1,2,3))

        dice = (
            2 * intersection + 1e-6
        ) / (
            preds.sum((1,2,3))
            + masks.sum((1,2,3))
            + 1e-6
        )

        dice_scores.extend(
            dice.cpu().numpy()
        )

print(
    "Mean Dice:",
    np.mean(dice_scores)
)

In [ ]:
import rasterio
from rasterio.windows import Window
import numpy as np
import torch
from tqdm import tqdm

NEW_TIF = "/content/drive/MyDrive/road_segmentation/1.tif"

TILE_SIZE = 512

with rasterio.open(NEW_TIF) as src:

    height = src.height
    width = src.width

    prediction_mask = np.zeros(
        (height, width),
        dtype=np.uint8
    )

    model.eval()

    for y in tqdm(range(0, height, TILE_SIZE)):
        for x in range(0, width, TILE_SIZE):

            if (
                x + TILE_SIZE > width
                or
                y + TILE_SIZE > height
            ):
                continue

            window = Window(
                x,
                y,
                TILE_SIZE,
                TILE_SIZE
            )

            tile = src.read(
                [1,2,3],
                window=window
            )

            tile = tile.transpose((1,2,0))

            tile = tile.astype(np.float32) / 255.0

            tile = (
                torch.tensor(tile)
                .permute(2,0,1)
                .unsqueeze(0)
                .to(device)
            )

            with torch.no_grad():

                pred = model(tile)

                pred = torch.sigmoid(pred)

                pred = (
                    pred > 0.5
                ).float()

            pred = (
                pred
                .squeeze()
                .cpu()
                .numpy()
            )

            prediction_mask[
                y:y+TILE_SIZE,
                x:x+TILE_SIZE
            ] = pred * 255

In [ ]:
with rasterio.open(NEW_TIF) as src:

    profile = src.profile

profile.update(
    count=1,
    dtype=rasterio.uint8
)

with rasterio.open(
    "predicted_mask.tif",
    "w",
    **profile
) as dst:

    dst.write(
        prediction_mask,
        1
    )

print("Saved predicted_mask.tif")

In [ ]:
import matplotlib.pyplot as plt
import random

# Get dimensions of the prediction mask
height, width = prediction_mask.shape

# Define a crop size, e.g., 2000x2000 pixels
CROP_SIZE = 2000

# Ensure CROP_SIZE does not exceed image dimensions
CROP_SIZE = min(CROP_SIZE, height, width)

# Calculate maximum possible starting coordinates for a valid crop
max_y = height - CROP_SIZE
max_x = width - CROP_SIZE

# Generate random starting coordinates
# If image dimensions are smaller than CROP_SIZE, start from 0
start_y = random.randint(0, max_y) if max_y > 0 else 0
start_x = random.randint(0, max_x) if max_x > 0 else 0

crop = prediction_mask[
    start_y : start_y + CROP_SIZE,
    start_x : start_x + CROP_SIZE
]

plt.figure(figsize=(8,8))
plt.imshow(crop, cmap="gray")
plt.title(f"Predicted Roads (Crop from {start_y},{start_x} to {start_y+CROP_SIZE},{start_x+CROP_SIZE})")
plt.show()

In [ ]:
import matplotlib.pyplot as plt

with rasterio.open(NEW_TIF) as src:

    image = src.read([1,2,3])

image = image.transpose((1,2,0))

# Use the same random crop coordinates as the previous cell
crop_img = image[start_y : start_y + CROP_SIZE, start_x : start_x + CROP_SIZE]
crop_mask = prediction_mask[start_y : start_y + CROP_SIZE, start_x : start_x + CROP_SIZE]

plt.figure(figsize=(15,7))

plt.subplot(1,2,1)
plt.imshow(crop_img)
plt.title(f"Satellite Image Crop ({start_y},{start_x})")

plt.subplot(1,2,2)
plt.imshow(crop_img)
plt.imshow(
    crop_mask,
    alpha=0.5, # Increased alpha for better visibility
    cmap="Reds"
)
plt.title(f"Road Predictions Overlay (Crop from {start_y},{start_x} to {start_y+CROP_SIZE},{start_x+CROP_SIZE})")

plt.tight_layout()
plt.show()